# Smart Home Energy Simulation

This notebook provides an interactive object-oriented model for energy simulation. You can define components like PV systems, batteries, and heat pumps, and then run simulations to analyze energy balance and costs.

## 1. Simulation Logic
Define the base classes for components and the simulation engine.

In [46]:
import importlib, simulation
importlib.reload(simulation)
from simulation import *
import pandas as pd, numpy as np
print('simulation.py loaded.')


simulation.py loaded.


## 2. Setup & Run Baseline
Initialize and execute the **Base Case**.

In [47]:
from simulation import setup_base, run_scenario, show_comparison
import pandas as pd, numpy as np

# Default duration — old scenario cells (Cell 6/8) use these variables.
# They get overridden when run_scenario() is called with explicit duration_days.
duration_days = 1
steps_per_day = 4
dt            = 24.0 / steps_per_day
total_steps   = steps_per_day * duration_days

print(f'Ready. Default: {duration_days} day(s), {total_steps} steps x {dt:.1f} h each.')


Ready. Default: 1 day(s), 4 steps x 6.0 h each.


## 3. Scenario Adjustment
Define multipliers for the scenario.

In [48]:
pv_factor = 0.0     # 0% PV (no solar available)
battery_factor = 0.2 # 80% reduction
print(f"FACTORS: PV x{pv_factor}, Battery x{battery_factor}")

FACTORS: PV x0.0, Battery x0.2


## 4. Run Scenario
Execute the simulation with the adjusted parameters.

In [49]:
# ── First Scenario (no PV, reduced battery) ──────────────────
base_summary, scenario_summary = run_scenario(
    duration_days  = duration_days,
    season         = 'winter',
    pv_factor      = pv_factor,
    battery_factor = battery_factor,
)
print(f'SCENARIO RUN: {duration_days} day(s)')
for k, v in scenario_summary.items(): print(f'  {k:28}: {v:8.2f}')



--------------------------------------------------
CURRENT SYSTEM PARAMETERS
--------------------------------------------------
Heat        : Avg Demand=2.875 
Elec        : Avg Demand=1.125 
Cooling     : Avg Demand=0.0   
PV          : Capacity=0.0    Efficiency=1.0   
HeatPump    : Capacity=3.0    Efficiency=3.5   
GasBoiler   : Capacity=5.0    Efficiency=0.9   
Chiller     : Capacity=4.0    Efficiency=3.0   
Battery     : Capacity=2.0    MaxCharge=5.0   
HeatStorage : Capacity=5.0    MaxCharge=3.0   
Grid        : ImportPrice=0.3    ExportPrice=0.1   
--------------------------------------------------

SCENARIO RUN: 1 day(s)
  Total Cost (CHF)            :    14.16
  Total Emissions (kg CO2)    :    18.88
  PV Generation (kWh)         :     0.00
  Heat Demand (kWh)           :    69.00
  Heat Supplied (kWh)         :    69.00
  Cooling Demand (kWh)        :     0.00
  Cooling Supplied (kWh)      :     0.00
  Elec Demand (kWh)           :    45.00
  Elec Supplied (kWh)         :   

## 5. Comparison Matrix
Visualize the analytics: **SCR** (Self-Consumption), **SSR** (Self-Sufficiency), and **Comfort**.

In [50]:
import pandas as pd, numpy as np

def calculate_comp(base, scen):
    res = []
    for m in base:
        b, s = base[m], scen[m]
        dev = ((s-b)/abs(b)*100) if b != 0 else (0.0 if s == 0 else float('nan'))
        res.append({'Metric': m, 'Base': b, 'Scenario': s, 'Rel. Dev. (%)': dev})
    return pd.DataFrame(res)

def color_dev(v):
    if pd.isna(v) or v == 0: return 'background-color:#3b82f6;color:white'
    return 'background-color:#22c55e;color:white' if v > 0 else 'background-color:#ef4444;color:white'

def show_comparison(base, scen, title='Comparison'):
    print(f'--- {title} ---')
    df = calculate_comp(base, scen)
    return df.style.map(color_dev, subset=['Rel. Dev. (%)']).format(
        {'Base': '{:.2f}', 'Scenario': '{:.2f}', 'Rel. Dev. (%)': '{:+.2f}%'})


## Heat Wave
set parameters to Heat Wave Scneario

In [51]:
# ════════════════════════════════════════════════════════════
# SCENARIO  —  edit values, then run this cell
# ════════════════════════════════════════════════════════════
base_summary, scenario_summary = run_scenario(
    duration_days           = 9,
    season                  = 'summer',  # <--- set season to summer
    pv_factor               = 0.60,
    battery_factor          = 0.90,
    heat_demand_factor      = 1.0,   # 2.0 = coldspell
    cooling_demand_factor   = 2.0,   # 2.0 = heatwave
    elec_demand_factor      = 1.0,
    hp_capacity_factor      = 1.0,
    chiller_capacity_factor = 1.0,
    boiler_capacity_factor  = 1.0,
    heat_storage_factor     = 1.0,
    grid_import_price_factor = 1.0,
    grid_export_price_factor = 1.0,
)
print('=== BASE (Summer) ===')
for k, v in base_summary.items():     print(f'  {k:28}: {v:8.2f}')
print('=== SCENARIO (Heatwave) ===')
for k, v in scenario_summary.items(): print(f'  {k:28}: {v:8.2f}')



--------------------------------------------------
CURRENT SYSTEM PARAMETERS
--------------------------------------------------
Heat        : Avg Demand=0.35  
Elec        : Avg Demand=1.125 
Cooling     : Avg Demand=4.0   
PV          : Capacity=3.0    Efficiency=1.0   
HeatPump    : Capacity=3.0    Efficiency=3.5   
GasBoiler   : Capacity=5.0    Efficiency=0.9   
Chiller     : Capacity=4.0    Efficiency=3.0   
Battery     : Capacity=9.0    MaxCharge=5.0   
HeatStorage : Capacity=5.0    MaxCharge=3.0   
Grid        : ImportPrice=0.3    ExportPrice=0.1   
--------------------------------------------------

=== BASE (Summer) ===
  Total Cost (CHF)            :    -8.36
  Total Emissions (kg CO2)    :     1.06
  PV Generation (kWh)         :   513.00
  Heat Demand (kWh)           :    75.60
  Heat Supplied (kWh)         :    75.60
  Cooling Demand (kWh)        :   432.00
  Cooling Supplied (kWh)      :   432.00
  Elec Demand (kWh)           :   412.10
  Elec Supplied (kWh)         :   4

In [52]:
show_comparison(base_summary, scenario_summary, 'Heatwave vs Base')


--- Heatwave vs Base ---


,Metric,Base,Scenario,Rel. Dev. (%)
0,Total Cost (CHF),-8.36,56.70,+778.44%
1,Total Emissions (kg CO2),1.06,75.60,+7051.75%
2,PV Generation (kWh),513.00,307.80,-40.00%
3,Heat Demand (kWh),75.60,75.60,+0.00%
4,Heat Supplied (kWh),75.60,75.60,+0.00%
5,Cooling Demand (kWh),432.00,864.00,+100.00%
6,Cooling Supplied (kWh),432.00,702.00,+62.50%
7,Elec Demand (kWh),412.10,501.51,+21.70%
8,Elec Supplied (kWh),412.10,501.51,+21.70%
9,Grid Import (kWh),2.64,189.01,+7051.75%


## Cold Spell Scenario
Set parameters for a cold spell event (low temperatures, high heat demand, reduced PV).

In [53]:
# ════════════════════════════════════════════════════════════
# COLD SPELL SCENARIO  —  edit values, then run this cell
# ════════════════════════════════════════════════════════════
base_summary_cs, scenario_summary_cs = run_scenario(
    duration_days           = 4,     # e.g. 4-day cold spell
    season                  = 'winter',  # <--- set season to winter
    pv_factor               = 0.3,   # less sun in winter
    battery_factor          = 1.0,
    heat_demand_factor      = 2.0,   # double heat demand
    cooling_demand_factor   = 0.0,   # no cooling needed
    elec_demand_factor      = 1.2,   # slightly more elec (lights etc.)
    hp_capacity_factor      = 1.0,
    chiller_capacity_factor = 1.0,
    boiler_capacity_factor  = 1.0,
    heat_storage_factor     = 1.0,
    grid_import_price_factor = 1.0,
    grid_export_price_factor = 1.0,
)
print('=== BASE (Winter) ===')
for k, v in base_summary_cs.items():     print(f'  {k:28}: {v:8.2f}')
print('=== COLD SPELL SCENARIO ===')
for k, v in scenario_summary_cs.items(): print(f'  {k:28}: {v:8.2f}')



--------------------------------------------------
CURRENT SYSTEM PARAMETERS
--------------------------------------------------
Heat        : Avg Demand=5.75  
Elec        : Avg Demand=1.3499999999999999
Cooling     : Avg Demand=0.0   
PV          : Capacity=1.5    Efficiency=1.0   
HeatPump    : Capacity=3.0    Efficiency=3.5   
GasBoiler   : Capacity=5.0    Efficiency=0.9   
Chiller     : Capacity=4.0    Efficiency=3.0   
Battery     : Capacity=10.0   MaxCharge=5.0   
HeatStorage : Capacity=5.0    MaxCharge=3.0   
Grid        : ImportPrice=0.3    ExportPrice=0.1   
--------------------------------------------------

=== BASE (Winter) ===
  Total Cost (CHF)            :    11.84
  Total Emissions (kg CO2)    :    15.86
  PV Generation (kWh)         :   156.00
  Heat Demand (kWh)           :   276.00
  Heat Supplied (kWh)         :   276.00
  Cooling Demand (kWh)        :     0.00
  Cooling Supplied (kWh)      :     0.00
  Elec Demand (kWh)           :   180.86
  Elec Supplied (kWh)  

### Cold Spell Comparison Matrix

In [54]:
show_comparison(base_summary_cs, scenario_summary_cs, 'Cold Spell vs Base')


--- Cold Spell vs Base ---


,Metric,Base,Scenario,Rel. Dev. (%)
0,Total Cost (CHF),11.84,91.82,+675.53%
1,Total Emissions (kg CO2),15.86,122.42,+671.85%
2,PV Generation (kWh),156.00,46.80,-70.00%
3,Heat Demand (kWh),276.00,552.00,+100.00%
4,Heat Supplied (kWh),276.00,552.00,+100.00%
5,Cooling Demand (kWh),0.00,0.00,+0.00%
6,Cooling Supplied (kWh),0.00,0.00,+0.00%
7,Elec Demand (kWh),180.86,211.89,+17.16%
8,Elec Supplied (kWh),180.86,211.89,+17.16%
9,Grid Import (kWh),26.32,159.39,+505.61%
